# Experiment 10: 95% Wilson Confidence Intervals (E2)

**Reviewer concern (R1):** §14 reports point estimates without CIs. R1 asked us to report 95% Wilson intervals on every refusal/ASR rate.

**This notebook:** consumes the existing §14 result JSONs (`exp1_*.json`, `exp4_*.json`, `exp5_*.json`, etc.) and produces a single aggregated JSON with point estimate + Wilson 95% CI for every reported proportion. Local-only: no GPU needed.

**Output:** `experiments/results/section14_intervals.json` and a CSV summary for the manuscript.


## Setup

In [ ]:
%%capture
!pip install statsmodels pandas -q
import json, glob
from pathlib import Path
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportion_confint


In [ ]:
# Local fallback: if running outside Colab, point at the local results dir
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = Path('/content/drive/MyDrive/PhD/paper1-survey/experiments/results')
except ImportError:
    RESULTS_DIR = Path('experiments/results')
print('Reading from', RESULTS_DIR)


## Wilson interval helper

In [ ]:
def wilson(k: int, n: int, alpha: float = 0.05) -> tuple[float, float, float]:
    if n == 0:
        return 0.0, 0.0, 0.0
    p = k / n
    lo, hi = proportion_confint(k, n, alpha=alpha, method='wilson')
    return p, lo, hi

# Sanity check
assert wilson(0, 0) == (0.0, 0.0, 0.0)
p, lo, hi = wilson(50, 100)
print(f'50/100 -> p={p:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]')


## Aggregate every proportion in §14

Each existing JSON in `results/` carries enough information (numerator and denominator implied by the `results` array) to recompute proportions with intervals. We extract:

- Harmful refusal rate per model (Exp 1)
- Benign refusal rate per model (Exp 1)
- Multilingual safety scores (Exp 4)
- Jailbreak refusal rates (Exp 5)
- Multi-turn refusal rates (Exp 6)
- Compression-aware refusal rates (Exp 2)


In [ ]:
agg = {}
for path in sorted(RESULTS_DIR.glob('exp1_*.json')):
    d = json.loads(path.read_text())
    model = d.get('model', path.stem)
    rs = d.get('results', [])
    harmful = [r for r in rs if r.get('split') == 'harmful']
    benign = [r for r in rs if r.get('split') == 'benign']

    h_n = len(harmful)
    h_k = sum(1 for r in harmful if r.get('llama_guard_label') == 'safe')
    b_n = len(benign)
    b_k = sum(1 for r in benign if r.get('llama_guard_label') != 'safe')  # benign refusal = flagged

    h_p, h_lo, h_hi = wilson(h_k, h_n)
    b_p, b_lo, b_hi = wilson(b_k, b_n)
    agg[model] = {
        'harmful_refusal': {'p': h_p, 'ci_lo': h_lo, 'ci_hi': h_hi, 'n': h_n, 'k': h_k},
        'benign_refusal': {'p': b_p, 'ci_lo': b_lo, 'ci_hi': b_hi, 'n': b_n, 'k': b_k},
        'safety_score': h_p - b_p,
    }

summary = pd.DataFrame.from_dict(agg, orient='index')
print(summary.head())


## Manuscript-ready summary CSV

Produces one row per model with columns suitable for direct paste into Table~\\ref{tab:empirical} after CI annotation.


In [ ]:
rows = []
for model, m in agg.items():
    rows.append({
        'model': model,
        'harmful_p': m['harmful_refusal']['p'],
        'harmful_ci': f'[{m["harmful_refusal"]["ci_lo"]:.3f}, {m["harmful_refusal"]["ci_hi"]:.3f}]',
        'benign_p': m['benign_refusal']['p'],
        'benign_ci': f'[{m["benign_refusal"]["ci_lo"]:.3f}, {m["benign_refusal"]["ci_hi"]:.3f}]',
        'safety_score': m['safety_score'],
    })
summary_df = pd.DataFrame(rows)
out_csv = RESULTS_DIR / 'section14_intervals.csv'
out_json = RESULTS_DIR / 'section14_intervals.json'
summary_df.to_csv(out_csv, index=False)
out_json.write_text(json.dumps(agg, indent=2))
print(f'Saved {out_csv} and {out_json}')
